# 实验 1：解剖 Qwen3-0.6B-Base 的静态结构

## 目标

像拆开一台机器一样，观察这个模型由哪些部件组成，以及每个部件的尺寸。

本实验会加载约 1.2GB 的原始权重，但**不生成文本、不训练模型**。

```text
token IDs
  ↓ embed_tokens
1024 维内部向量
  ↓ 28 个 Transformer block
结合上下文后的 1024 维向量
  ↓ lm_head
151,936 个候选下一个 token 的分数
```


In [1]:
from pathlib import Path

import torch
import transformers
from transformers import AutoModelForCausalLM


def find_project_root() -> Path:
    for directory in (Path.cwd(), *Path.cwd().parents):
        if (directory / 'models' / 'Qwen3-0.6B-Base').is_dir():
            return directory
    raise FileNotFoundError('找不到 models/Qwen3-0.6B-Base')


PROJECT_ROOT = find_project_root()
MODEL_PATH = PROJECT_ROOT / 'models' / 'Qwen3-0.6B-Base'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'模型目录: {MODEL_PATH}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')


项目根目录: /home/linjunjie/Workspace/xxdw1
模型目录: /home/linjunjie/Workspace/xxdw1/models/Qwen3-0.6B-Base
PyTorch: 2.13.0+cu130
Transformers: 5.15.1


In [2]:
# 本实验只观察结构，因此固定在 CPU 上加载。
# 这样不受 GPU Triton 编译器环境的影响。
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).cpu().eval()

print(f'模型类: {type(model).__name__}')
print(f'模型设备: {next(model.parameters()).device}')
print(f'权重数据类型: {next(model.parameters()).dtype}')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

模型类: Qwen3ForCausalLM
模型设备: cpu
权重数据类型: torch.bfloat16


## 顶层骨架

`Qwen3ForCausalLM` 是完整的“预测下一个 token”模型。它的主体 `model` 负责把 token 的内部向量加工 28 次；`lm_head` 则把最终的 1024 维向量映射到整个词表。

注意：以下所有 block 的主通道输入和输出都是 1024 维。这是模型内部信息流动的固定宽度。


In [3]:
config = model.config

print(f'总参数量: {sum(parameter.numel() for parameter in model.parameters()):,}')
print(f'词表大小: {config.vocab_size:,}')
print(f'隐藏向量维度: {config.hidden_size}')
print(f'Transformer block 数量: {config.num_hidden_layers}')
print(f'注意力头: Q={config.num_attention_heads}, KV={config.num_key_value_heads}')
print(f'每头维度: {config.head_dim}')
print(f'MLP 中间维度: {config.intermediate_size}')

print('\nQwen3ForCausalLM')
print('├── model: Qwen3Model')
print(f'│   ├── embed_tokens: Embedding({config.vocab_size}, {config.hidden_size})')
print(f'│   ├── layers: ModuleList × {len(model.model.layers)}')
print(f'│   └── norm: {model.model.norm}')
print(f'└── lm_head: Linear({config.hidden_size}, {config.vocab_size}, bias=False)')


总参数量: 596,049,920
词表大小: 151,936
隐藏向量维度: 1024
Transformer block 数量: 28
注意力头: Q=16, KV=8
每头维度: 128
MLP 中间维度: 3072

Qwen3ForCausalLM
├── model: Qwen3Model
│   ├── embed_tokens: Embedding(151936, 1024)
│   ├── layers: ModuleList × 28
│   └── norm: Qwen3RMSNorm((1024,), eps=1e-06)
└── lm_head: Linear(1024, 151936, bias=False)


## 把 PDF 总览图与 Python 模块树对照起来

`1.pdf` 是**计算流程视角**：它回答“数据依次经过哪些阶段”。而上一个代码单元打印出的树状结构是**程序对象视角**：它回答“这些阶段在 Transformers 模型对象中分别挂在哪里”。两种视角描述的是同一个模型，建议并排或上下对照阅读。

<div align="center">
  <img src="../assets/qwen3-backbone-overview.png" alt="Qwen3 顶层结构总览图" width="700" style="max-width: 100%; border: 1px solid #cbd5e1; border-radius: 8px;">
</div>

原始 PDF 仍可通过下方链接打开：

[打开 1.pdf：Backbone Model 总览图](../1.pdf)

### 两张图如何对应

| PDF 中的模块 | Python 树状结构中的对象 | 作用 |
|---|---|---|
| Input Token | `model.model.embed_tokens` 的输入 | token ID 进入模型 |
| Embedding | `model.model.embed_tokens` | 把 token ID 查成 1024 维向量 |
| N × Transformer Block | `model.model.layers`，其中 `N = 28` | 反复加工并结合上下文 |
| Final RMSNorm | `model.model.norm` | 对最后的 hidden states 做最终归一化 |
| LM Head | `model.lm_head` | 将 1024 维向量映射到词表维度 |
| Output Logits | `model.lm_head` 的输出 | 每个候选下一个 token 的分数 |

注意：PDF 里的方框是“功能阶段”，树状文本里的缩进是“对象包含关系”。例如 `layers: ModuleList × 28` 在树中是一行，但在 PDF 中展开后代表 28 个连续的 Transformer block。


## 拆开一个 Transformer block

28 个 block 的结构相同，所以先观察第 0 层就够了。

一个 block 的主路径是：

```text
输入 → RMSNorm → Self-Attention → 残差相加
     → RMSNorm → MLP            → 残差相加 → 输出
```

这里的 Self-Attention 让不同 token 交换信息；MLP 则单独加工每个 token 的内部特征。


In [7]:
layer = model.model.layers[0]
print(layer)


Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
    (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
    (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
    (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
)


## 关键权重矩阵

PyTorch 的线性层权重形状写作 `(输出维度, 输入维度)`。例如 Q 投影的 `(2048, 1024)` 表示：每个 1024 维输入向量会被变换为 2048 维 Q 向量。

Qwen3-0.6B 使用 GQA（Grouped-Query Attention）：16 个 Query 头，但只有 8 个 Key 头和 8 个 Value 头。因为每头 128 维，所以 Q 是 `16 × 128 = 2048` 维，而 K/V 各是 `8 × 128 = 1024` 维。


In [9]:
parameters = (
    ('token embedding', model.model.embed_tokens.weight),
    ('Q projection', layer.self_attn.q_proj.weight),
    ('K projection', layer.self_attn.k_proj.weight),
    ('V projection', layer.self_attn.v_proj.weight),
    ('attention output projection', layer.self_attn.o_proj.weight),
    ('MLP gate projection', layer.mlp.gate_proj.weight),
    ('MLP up projection', layer.mlp.up_proj.weight),
    ('MLP down projection', layer.mlp.down_proj.weight),
    ('language-model head', model.lm_head.weight),
)

for name, parameter in parameters:
    print(f'{name:28} {tuple(parameter.shape)}')

shared_weights = (
    model.model.embed_tokens.weight.data_ptr()
    == model.lm_head.weight.data_ptr()
)
print(f'\n输入 embedding 与 lm_head 是否共享同一块权重: {shared_weights}')


token embedding              (151936, 1024)
Q projection                 (2048, 1024)
K projection                 (1024, 1024)
V projection                 (1024, 1024)
attention output projection  (1024, 2048)
MLP gate projection          (3072, 1024)
MLP up projection            (3072, 1024)
MLP down projection          (1024, 3072)
language-model head          (151936, 1024)

输入 embedding 与 lm_head 是否共享同一块权重: True


## 小结与思考

你已经从真实权重中确认：Qwen3-0.6B 有 28 个重复 block，内部主通道宽度是 1024，attention 使用 16 个 Q 头与 8 个 KV 头，MLP 会把 1024 维暂时扩展到 3072 维再压回 1024 维。

试着回答这三个问题：

1. 为什么 28 个 block 都保持输入和输出为 1024 维，而不是每层都继续变宽？
2. 为什么 Q 是 2048 维，但 K 和 V 各只有 1024 维？
3. 为什么模型可以让输入 embedding 和输出 lm_head 共享同一块词表矩阵？

下一实验会亲手观察一个 token ID 如何从 `embed_tokens` 变成 1024 维向量。
